# Explore Vector Database recipes.lance

## 1. Retrieval from Vector Database

In this step, we test semantic search using the vector database (LanceDB).

The goal is to retrieve the most relevant recipes given a user query.

In [3]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

import lancedb
from sentence_transformers import SentenceTransformer
from src.cookmate.utils.config import DB_DIR

## 2. Load embedding model

We use the same embedding model used during ingestion to ensure consistency.

In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7072.45it/s]


## 3. Connect to LanceDB

We connect to the local vector database and open the recipes table.

In [5]:
db = lancedb.connect(DB_DIR)
table = db.open_table("recipes")

## 4. Define user query

We simulate a user query to test semantic retrieval.

In [6]:
query = "chicken pasta with tomato"

## 5. Convert query to embedding

We transform the query into a vector representation.

In [7]:
query_vector = model.encode(query)

## 6. Perform similarity search

We retrieve the most similar recipes from the database.

In [8]:
result = table.search(query_vector).limit(3).to_pandas()
result

,id,text,vector,_distance
0,recipe_1735,Recipe: Chicken Spaghetti. Ingredients: stalks...,"[-0.058917366, -0.058045264, 0.006024487, -0.0...",0.516375
1,recipe_1635,"Recipe: Chicken, Pesto And Sun-Dried Tomato Pa...","[-0.08670863, -0.05756682, 0.027218172, -0.009...",0.572196
2,recipe_1065,Recipe: Pasta In Tomato Cream Sauce. Ingredien...,"[-0.068504214, -0.032079052, 0.026498469, 0.00...",0.642061


## 7. Results

The retrieved recipes are semantically similar to the query.
This demonstrates that the system can find relevant recipes even without exact keyword matching.
The distance metric is also quite good.

In this experiment, we use local embeddings for ingestion to avoid rate limits.
We tested Cohere embeddings, but they could not perform well due to API limitations and rate limits.

It seems that in production we could switch to Cohere embeddings to improve semantic accuracy.
However, considering the relatively small size of the dataset, the current approach already provides good retrieval results.

We may use Cohere embeddings at query time for higher semantic accuracy.